In [ ]:
import pandas as pd
import numpy as np
from skimage.metrics import structural_similarity as ssim
import torch
import os
import sys
import numpy as np

root_path = os.path.abspath(os.path.join('..'))
if root_path not in sys.path:
    sys.path.append(root_path)
 
from syn_project.utils_train import *
from syn_project.utils_color_analysis import *
from syn_project.utils_notebook import *

# ---- helpers ----

def compute_reconstruction_quality(original_rgb, decoded_rgb):
    """
    original_rgb, decoded_rgb : tensors ou numpy (N, H, W, 3) ou (N, 3, H, W), valeurs dans [0,1].
    Retourne dict avec mse, ssim_mean.
    """
    # → numpy (N, H, W, 3) dans [0,1]
    def to_nhwc(x):
        if isinstance(x, torch.Tensor):
            x = x.detach().cpu().numpy()
        if x.ndim == 4 and x.shape[1] == 3:   # (N,3,H,W)
            x = x.transpose(0, 2, 3, 1)
        return x.astype(np.float32)

    orig = to_nhwc(original_rgb)
    dec  = to_nhwc(decoded_rgb)

    mse  = float(np.mean((orig - dec) ** 2))
    ssim_scores = [
        ssim(orig[i], dec[i], channel_axis=-1, data_range=1.0)
        for i in range(len(orig))
    ]
    return {"mse": mse, "ssim": float(np.mean(ssim_scores))}


# ---- boucle principale ----

conditions = [
    "3mod_basic_s126_a05",
    "3mod_basic_s126_no_cont",
    "3mod_basic_s126_low_cont",
    "3mod_basic_s126_no_trans",
    "3mod_basic_s126_low_trans",
    "3mod_basic_s126_no_cy",
    "3mod_basic_s126_low_cy",
    "3mod_basic_s126_no_dcy",
    "3mod_basic_s126_low_dcy",
]

checkpoint_epoch  = 0
n_samples_test    = 1000
split             = "test"
dataset           = "biased_00"

rows = []

for condition in conditions:
    print(f"\n=== {condition} ===")
    with total_silence():
        (global_workspace, domain_mods, gw_mod,
         visual_module, original_data,
         latent_domains, modules_name) = get_modules_data_from_exp(
            experiment_name=condition,
            n_samples_test=n_samples_test,
            split=split,
            checkpoint_epoch=checkpoint_epoch,
        )

    objects = get_objects_from_v_latents(
        latent_domains, gw_mod, global_workspace, modules_name,
        modality_from='attr', modality_through='color',
        modality_main=['attr'], modality_add='color',
        start_v=True,
    )

    original_images_rgb = visual_module.decode_images(original_data['v_latents'])

    cat = []
    if 'attr' in modules_name:
        cat = original_data['attr'][0]
    if 'cat' in modules_name:
        cat = original_data['cat']

    # vision4 = la reconstruction finale qui t'intéresse
    decoded_images = visual_module.decode_images(objects['vision4'])

    # --- qualité de reconstruction ---
    recon = compute_reconstruction_quality(original_images_rgb, decoded_images)

    # --- analyse couleur / LDA ---
    colors_np = objects['x2']['color'].detach().cpu().numpy()
    cats      = cat.argmax(dim=1).detach().cpu().numpy()
    metrics, _ = hue_analysis(colors_np, cats,
                               cat_names=CAT_NAMES,
                               value=0.75, saturation_boost=1.8)
    results = logistic_probe(colors_np, cats)

    rows.append({
        "condition"        : condition,
        "mse"              : recon["mse"],
        "ssim"             : recon["ssim"],
        "lda_score"        : metrics["lda_score"],
        "probe_accuracy"   : results["accuracy"],
    })

    del decoded_images, objects   # libère la mémoire GPU
    torch.cuda.empty_cache()

# ---- tableau récapitulatif ----

df = pd.DataFrame(rows).set_index("condition")
df = df.sort_values("lda_score", ascending=False)   # tri par LDA, ajuste si besoin

# affichage Jupyter
display(
    df.style
      .format({"mse": "{:.4f}", "ssim": "{:.3f}",
               "lda_score": "{:.3f}", "probe_accuracy": "{:.2%}"})
      .background_gradient(subset=["ssim", "lda_score", "probe_accuracy"], cmap="RdYlGn")
      .background_gradient(subset=["mse"], cmap="RdYlGn_r")   # MSE : plus bas = mieux
)


=== 3mod_basic_s126_a05 ===


/home/lucas/.cache/pypoetry/virtualenvs/alexis-n7zQ69N0-py3.11/lib/python3.11/site-packages/torch/nn/modules/module.py:1736: UserWarning: Implicit dimension choice for softmax has been deprecated. Change the call to include dim=X as an argument.
  return self._call_impl(*args, **kwargs)



=== 3mod_basic_s126_no_cont ===


/home/lucas/.cache/pypoetry/virtualenvs/alexis-n7zQ69N0-py3.11/lib/python3.11/site-packages/torch/nn/modules/module.py:1736: UserWarning: Implicit dimension choice for softmax has been deprecated. Change the call to include dim=X as an argument.
  return self._call_impl(*args, **kwargs)



=== 3mod_basic_s126_low_cont ===


...


=== 3mod_basic_s126_no_dcy ===


/home/lucas/.cache/pypoetry/virtualenvs/alexis-n7zQ69N0-py3.11/lib/python3.11/site-packages/torch/nn/modules/module.py:1736: UserWarning: Implicit dimension choice for softmax has been deprecated. Change the call to include dim=X as an argument.
  return self._call_impl(*args, **kwargs)



=== 3mod_basic_s126_low_dcy ===


/home/lucas/.cache/pypoetry/virtualenvs/alexis-n7zQ69N0-py3.11/lib/python3.11/site-packages/torch/nn/modules/module.py:1736: UserWarning: Implicit dimension choice for softmax has been deprecated. Change the call to include dim=X as an argument.
  return self._call_impl(*args, **kwargs)


,mse,ssim,lda_score,probe_accuracy
condition,,,,
3mod_basic_s126_a05,0.0097,0.860,0.993,99.00%
3mod_basic_s126_no_dcy,0.0114,0.818,0.949,96.50%
3mod_basic_s126_low_trans,0.0107,0.829,0.943,95.00%
3mod_basic_s126_low_dcy,0.0092,0.832,0.909,89.50%
3mod_basic_s126_low_cont,0.0093,0.860,0.755,88.00%
3mod_basic_s126_no_cont,0.0133,0.814,0.744,97.00%
3mod_basic_s126_no_trans,0.0173,0.747,0.452,48.50%
3mod_basic_s126_low_cy,0.0105,0.820,0.393,42.00%
3mod_basic_s126_no_cy,0.0100,0.820,0.386,38.00%
